# Test: Character Offset-Based Token Labeling

This notebook tests the **character offset approach** for labeling PLACE spans.

**Problem**: Word-boundary matching is loose and misses entities  
**Solution**: Use exact character offsets (`offset`/`end` columns) to identify tokens that overlap with annotations

We'll load real training data and demonstrate that offset-based matching correctly labels PLACE tokens.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from flair.data import Sentence
from collections import defaultdict

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

/Users/rikhoekstra/develop/republic_ner_matching/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Sample Annotations Data

Load the LOC training pairs from the parquet file. Each row contains:
- `paragraph_texts`: The full paragraph text
- `htr_span`: The entity text (what was annotated)
- `offset`: Character position where entity starts in the paragraph
- `canonical_entity`: The place name
- `date`: Date of the resolution

In [2]:
# Load training data
df = pd.read_parquet('data/training_pairs_loc_1626_1630_dedup.parquet')

print(f"Loaded {len(df)} training records")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")

# Show first few records
print(f"\nFirst 3 records:")
df.head(3)

Loaded 21310 training records

Columns: ['resolution_id', 'paragraph_texts', 'htr_span', 'canonical_entity', 'entity_type', 'entity_id', 'offset', 'date', 'source']

Data types:
resolution_id         str
paragraph_texts       str
htr_span              str
canonical_entity      str
entity_type           str
entity_id             str
offset              int64
date                  str
source                str
dtype: object

First 3 records:


,resolution_id,paragraph_texts,htr_span,canonical_entity,entity_type,entity_id,offset,date,source
0,session-3185-num-1-resolution-1,"[""Actere ende Resoltaren vande ho:Mo: heeren Staten Generael der Vereenichde Nederlanden."", ""den...",Engelant,Engeland,place,L0001860,205,1626-01-01,LOC-annotations
1,session-3185-num-1-resolution-1,"[""Actere ende Resoltaren vande ho:Mo: heeren Staten Generael der Vereenichde Nederlanden."", ""den...",Vranckryck,Frankrijk,place,L0007323,47,1626-01-01,LOC-annotations
2,session-3185-num-1-resolution-1,"[""Actere ende Resoltaren vande ho:Mo: heeren Staten Generael der Vereenichde Nederlanden."", ""den...",Vranckryck,Frankrijk,place,L0007323,95,1626-01-01,LOC-annotations


## 3. Extract Character Offsets from Annotations

Pick a sample resolution and extract all the PLACE annotations with their character offsets.

In [3]:
# Group by resolution_id and pick one
sample_res_id = df['resolution_id'].iloc[0]
sample_group = df[df['resolution_id'] == sample_res_id]

print(f"Resolution ID: {sample_res_id}")
print(f"Number of annotations in this resolution: {len(sample_group)}")
print(f"\nParagraph text (first 500 chars):")
para_text = sample_group.iloc[0]['paragraph_texts']
print(para_text[:500])
print(f"... ({len(para_text)} total chars)")

# Extract offsets
print(f"\n\n=== ANNOTATIONS (with character offsets) ===")
annotations = []
for idx, row in sample_group.iterrows():
    offset = row['offset']
    entity_text = row['htr_span']
    end = offset + len(entity_text)
    annotations.append({
        'offset': offset,
        'end': end,
        'text': entity_text,
        'canonical': row['canonical_entity']
    })
    print(f"Offset {offset:4d}-{end:4d}: '{entity_text}' → {row['canonical_entity']}")

print(f"\nTotal: {len(annotations)} PLACE annotations")
annotations_sorted = sorted(annotations, key=lambda x: x['offset'])
print(f"Sorted by offset: {[(a['offset'], a['text']) for a in annotations_sorted]}")

Resolution ID: session-3185-num-1-resolution-1
Number of annotations in this resolution: 5

Paragraph text (first 500 chars):
["Actere ende Resoltaren vande ho:Mo: heeren Staten Generael der Vereenichde Nederlanden.", "den xixen. deses om te besoigneren opte veilinge vande zee voor het aenstaende Somersaisoen, mitsgaders opte equippagie vande schepen die van wegens haer ho:Mo: gevougt sullen werden byde tweede Vloot van Engelant.", "Synde gerapporteert datte heer Ambassadeur van Vranckryck aengenomen heeft nieuwe debuoiren in Vranckryck te doen ten einde ordre gestelt mach werden, tot betalinge vande overgesondene wiss
... (720 total chars)


=== ANNOTATIONS (with character offsets) ===
Offset  205- 213: 'Engelant' → Engeland
Offset   47-  57: 'Vranckryck' → Frankrijk
Offset   95- 105: 'Vranckryck' → Frankrijk
Offset  249- 258: 'Amsterdam' → Amsterdam
Offset  323- 333: 'Vranckryck' → Frankrijk

Total: 5 PLACE annotations
Sorted by offset: [(47, 'Vranckryck'), (95, 'Vranckryck'), (205, 

## 4. Tokenize Text and Map to Offsets

Use Flair to tokenize the paragraph and compute character positions for each token.

In [5]:
# Tokenize with Flair
sentence = Sentence(para_text)

print(f"Tokenized into {len(sentence.tokens)} tokens")
print(f"\nFirst 15 tokens with their character positions:")
print(f"{'Idx':<4} {'Token':<20} {'Text':<30} {'Char Range':<15}")
print("-" * 70)

for i, token in enumerate(sentence.tokens[:15]):
    token_text = token.text
    # Flair doesn't automatically compute positions, so we need to do it
    # by finding the token in the text
    pos = para_text.find(token_text)
    print(f"{i:<4} {token_text:<20} (whitespace-split: '{token_text}') pos={pos}")

# Better approach: manually compute token positions by white-space splitting
words = para_text.split()
char_pos = 0
token_positions = []

for word_idx, word in enumerate(words):
    # Find this word in the text
    start_pos = para_text.find(word, char_pos)
    end_pos = start_pos + len(word)
    token_positions.append({
        'idx': word_idx,
        'text': word,
        'start': start_pos,
        'end': end_pos
    })
    char_pos = end_pos

print(f"\n\nRe-tokenized (by whitespace): {len(token_positions)} tokens")
print(f"\nFirst 15 tokens with character positions:")
print(f"{'Idx':<4} {'Token':<20} {'Start':<8} {'End':<8}")
print("-" * 40)
for tp in token_positions[:15]:
    print(f"{tp['idx']:<4} {tp['text']:<20} {tp['start']:<8} {tp['end']:<8}")

Tokenized into 123 tokens

First 15 tokens with their character positions:
Idx  Token                Text                           Char Range     
----------------------------------------------------------------------
0    ["                   (whitespace-split: '["') pos=0
1    Actere               (whitespace-split: 'Actere') pos=2
2    ende                 (whitespace-split: 'ende') pos=9
3    Resoltaren           (whitespace-split: 'Resoltaren') pos=14
4    vande                (whitespace-split: 'vande') pos=25
5    ho                   (whitespace-split: 'ho') pos=31
6    :                    (whitespace-split: ':') pos=33
7    Mo                   (whitespace-split: 'Mo') pos=34
8    :                    (whitespace-split: ':') pos=33
9    heeren               (whitespace-split: 'heeren') pos=38
10   Staten               (whitespace-split: 'Staten') pos=45
11   Generael             (whitespace-split: 'Generael') pos=52
12   der                  (whitespace-split: 'der') pos=61


## 5. Identify PLACE Spans Using Character Offsets

**Key idea**: A token belongs to a PLACE span if:
- `token.start <= annotation_end` AND `token.end >= annotation_start`

This is exact interval overlap, not word boundary matching.

In [6]:
# Apply character offset matching
labels = []
for token_pos in token_positions:
    token_start = token_pos['start']
    token_end = token_pos['end']
    token_text = token_pos['text']
    
    # Check if this token overlaps with ANY annotation
    is_place = False
    matching_annotations = []
    
    for annot in annotations_sorted:
        ann_start = annot['offset']
        ann_end = annot['end']
        
        # Check overlap: token_start < ann_end AND token_end > ann_start
        if token_start < ann_end and token_end > ann_start:
            is_place = True
            matching_annotations.append(annot['text'])
    
    label = 'PLACE' if is_place else 'O'
    labels.append({
        'idx': token_pos['idx'],
        'token': token_text,
        'start': token_start,
        'end': token_end,
        'label': label,
        'matched_annot': ', '.join(matching_annotations) if matching_annotations else ''
    })

# Show results
print("=== OFFSET-BASED LABELING RESULTS ===\n")
print(f"{'Idx':<5} {'Token':<20} {'Char Range':<20} {'Label':<8} {'Matched Annotations':<40}")
print("-" * 95)

for label_rec in labels[:30]:  # Show first 30
    char_range = f"{label_rec['start']}-{label_rec['end']}"
    print(f"{label_rec['idx']:<5} {label_rec['token']:<20} {char_range:<20} {label_rec['label']:<8} {label_rec['matched_annot']:<40}")

# Summary
place_count = sum(1 for l in labels if l['label'] == 'PLACE')
print(f"\n\n📊 SUMMARY:")
print(f"Total tokens: {len(labels)}")
print(f"PLACE tokens: {place_count}")
print(f"O tokens: {len(labels) - place_count}")

=== OFFSET-BASED LABELING RESULTS ===

Idx   Token                Char Range           Label    Matched Annotations                     
-----------------------------------------------------------------------------------------------
0     ["Actere             0-8                  O                                                
1     ende                 9-13                 O                                                
2     Resoltaren           14-24                O                                                
3     vande                25-30                O                                                
4     ho:Mo:               31-37                O                                                
5     heeren               38-44                O                                                
6     Staten               45-51                PLACE    Vranckryck                              
7     Generael             52-60                PLACE    Vranckryck              

## 6. Validate Offset-Based Matching

Verify the matching by reconstructing entity spans from labeled tokens and comparing with original annotations.

In [7]:
# Extract PLACE spans from labels
extracted_spans = []
current_span = None

for label_rec in labels:
    if label_rec['label'] == 'PLACE':
        if current_span is None:
            current_span = {
                'start': label_rec['start'],
                'end': label_rec['end'],
                'tokens': [label_rec['token']]
            }
        else:
            # Check if contiguous (or very close)
            if label_rec['start'] <= current_span['end'] + 1:
                current_span['end'] = label_rec['end']
                current_span['tokens'].append(label_rec['token'])
            else:
                extracted_spans.append(current_span)
                current_span = {
                    'start': label_rec['start'],
                    'end': label_rec['end'],
                    'tokens': [label_rec['token']]
                }
    else:
        if current_span is not None:
            extracted_spans.append(current_span)
            current_span = None

if current_span is not None:
    extracted_spans.append(current_span)

# Add extracted text
for span in extracted_spans:
    span['text'] = para_text[span['start']:span['end']]

print("=== EXTRACTED SPANS (from offset-based labels) ===\n")
print(f"{'Span':<30} {'Char Range':<20} {'Tokens':<40}")
print("-" * 90)
for span in extracted_spans:
    tokens_str = ' '.join(span['tokens'][:5])
    if len(span['tokens']) > 5:
        tokens_str += f" ... ({len(span['tokens'])} tokens)"
    char_range = f"{span['start']}-{span['end']}"
    print(f"{span['text']:<30} {char_range:<20} {tokens_str:<40}")

# Compare with original annotations
print("\n\n=== ORIGINAL ANNOTATIONS ===\n")
print(f"{'Entity':<30} {'Char Range':<20} {'Canonical':<40}")
print("-" * 90)
for annot in annotations_sorted:
    char_range = f"{annot['offset']}-{annot['end']}"
    print(f"{annot['text']:<30} {char_range:<20} {annot['canonical']:<40}")

# Check overlap
print("\n\n=== VALIDATION: Do extracted spans match original annotations? ===\n")
matches = 0
for annot in annotations_sorted:
    found = False
    for span in extracted_spans:
        # Check if they have the same character range (or close)
        if span['start'] == annot['offset'] and span['end'] == annot['end']:
            print(f"✓ MATCH: '{annot['text']}' (offset {annot['offset']}-{annot['end']})")
            found = True
            matches += 1
            break
    if not found:
        print(f"✗ MISS: '{annot['text']}' (offset {annot['offset']}-{annot['end']}) not found in extracted spans")

print(f"\n\nMatched: {matches}/{len(annotations_sorted)}")
if matches == len(annotations_sorted):
    print("✓ PERFECT: All annotations extracted correctly!")

=== EXTRACTED SPANS (from offset-based labels) ===

Span                           Char Range           Tokens                                  
------------------------------------------------------------------------------------------
Staten Generael                45-60                Staten Generael                         
"den xixen. deses              92-109               "den xixen. deses                       
equippagie                     202-212              equippagie                              
ho:Mo: gevougt                 247-261              ho:Mo: gevougt                          
gerapporteert datte            317-336              gerapporteert datte                     


=== ORIGINAL ANNOTATIONS ===

Entity                         Char Range           Canonical                               
------------------------------------------------------------------------------------------
Vranckryck                     47-57                Frankrijk                      

## 7. Compare Offset vs Word Matching Approaches

Demonstrate why **word boundary matching** (the old approach) fails and **character offset matching** works.

In [9]:
# Simulate OLD word-boundary matching approach
print("=== OLD APPROACH: Word Boundary Matching ===\n")

word_match_labels = []
for token_pos in token_positions:
    token_text = token_pos['text']
    is_place_old = False
    
    # Old approach: check if token text is IN any annotation text
    for annot in annotations_sorted:
        if token_text.lower() in annot['text'].lower():
            is_place_old = True
            break
    
    word_match_labels.append({
        'idx': token_pos['idx'],
        'token': token_text,
        'label_old': 'PLACE' if is_place_old else 'O'
    })

old_place_count = sum(1 for l in word_match_labels if l['label_old'] == 'PLACE')
print(f"Word matching found: {old_place_count} PLACE tokens")

# Compare side by side
print("\n=== SIDE-BY-SIDE COMPARISON ===\n")
print(f"{'Idx':<4} {'Token':<20} {'Offset Match':<15} {'Word Match':<15} {'Correct?':<10}")
print("-" * 65)

comparison_results = {
    'offset_correct': 0,
    'offset_wrong': 0,
    'word_correct': 0,
    'word_wrong': 0
}

# Find an interesting example (with disagreement if possible)
shown = 0
for i in range(len(labels)):
    if shown < 20:  # Show first 20
        offset_label = labels[i]['label']
        word_label = word_match_labels[i]['label_old']
        token_text = labels[i]['token']
        
        # Check which is actually correct
        correct_label = 'PLACE' if labels[i]['matched_annot'] else 'O'
        
        if offset_label == correct_label:
            comparison_results['offset_correct'] += 1
            offset_mark = "✓"
        else:
            comparison_results['offset_wrong'] += 1
            offset_mark = "✗"
        
        if word_label == correct_label:
            comparison_results['word_correct'] += 1
            word_mark = "✓"
        else:
            comparison_results['word_wrong'] += 1
            word_mark = "✗"
        
        # Show if there's disagreement or it's interesting
        if offset_label != word_label or offset_label == 'PLACE':
            print(f"{i:<4} {token_text:<20} {offset_label} ({offset_mark:<8}) {word_label} ({word_mark:<8}) {'DIFFER' if offset_label != word_label else 'SAME':<10}")
            shown += 1

print("\n\n=== ACCURACY ANALYSIS ===\n")
print(f"Offset-based approach:")
print(f"  ✓ Correct: {comparison_results['offset_correct']}")
print(f"  ✗ Wrong: {comparison_results['offset_wrong']}")
print(f"  Accuracy: {100 * comparison_results['offset_correct'] / (comparison_results['offset_correct'] + comparison_results['offset_wrong']):.1f}%")

print(f"\nWord-matching approach:")
print(f"  ✓ Correct: {comparison_results['word_correct']}")
print(f"  ✗ Wrong: {comparison_results['word_wrong']}")
if (comparison_results['word_correct'] + comparison_results['word_wrong']) > 0:
    print(f"  Accuracy: {100 * comparison_results['word_correct'] / (comparison_results['word_correct'] + comparison_results['word_wrong']):.1f}%")

print("\n\n🎯 CONCLUSION:")
print("The offset-based approach uses EXACT character positions from annotations.")
print("Word-matching is loose and error-prone (substring matching).")
print("For NER training, precision is critical → use character offsets!")

=== OLD APPROACH: Word Boundary Matching ===

Word matching found: 6 PLACE tokens

=== SIDE-BY-SIDE COMPARISON ===

Idx  Token                Offset Match    Word Match      Correct?  
-----------------------------------------------------------------
6    Staten               PLACE (✓       ) O (✗       ) DIFFER    
7    Generael             PLACE (✓       ) O (✗       ) DIFFER    
11   "den                 PLACE (✓       ) O (✗       ) DIFFER    
12   xixen.               PLACE (✓       ) O (✗       ) DIFFER    
13   deses                PLACE (✓       ) O (✗       ) DIFFER    
15   te                   O (✓       ) PLACE (✗       ) DIFFER    
27   equippagie           PLACE (✓       ) O (✗       ) DIFFER    
34   ho:Mo:               PLACE (✓       ) O (✗       ) DIFFER    
35   gevougt              PLACE (✓       ) O (✗       ) DIFFER    
44   gerapporteert        PLACE (✓       ) O (✗       ) DIFFER    
45   datte                PLACE (✓       ) O (✗       ) DIFFER    
49   Vranckr